# Olist E-Commerce &mdash; Data Preprocessing

**Turns the 9 raw Olist CSV tables into 9 clean, usable CSV tables.**

This notebook is fully self-contained and runnable end-to-end (just *Run All*).
It:

- keeps **all 9 tables** &mdash; no merging, no reducing the table count;
- keeps **every row** &mdash; the one exception is `geolocation`, whose byte-identical duplicate rows are removed (see below);
- **types every column** &mdash; dates &rarr; `datetime`, numbers &rarr; numeric, identifiers &rarr; string, enums &rarr; validated categories;
- reads **zip-code prefixes as strings** so leading zeros are preserved (e.g. `01037`);
- writes **ISO-formatted dates** and UTF-8 CSVs with no index column.

**Input:** `raw/` (9 original CSVs) &nbsp;&middot;&nbsp; **Output:** `cleaned/` (9 cleaned CSVs, written by this notebook).

> **Geolocation dedup:** 261,831 of `olist_geolocation_dataset.csv`&rsquo;s
> 1,000,163 rows (26.18%) are byte-identical copies of an earlier row. These are
> removed, leaving **738,332 unique rows** &mdash; every unique
> `(zip, lat, lng, city, state)` combination is kept exactly once.

In [1]:
from pathlib import Path
import pandas as pd

# Locate project root dynamically whether run from notebooks/ or project root
def find_project_root() -> Path:
    p = globals().get("__vsc_ipynb_file__")
    start = Path(p).resolve().parent if p else Path.cwd().resolve()
    for d in [start, *start.parents]:
        if (d / "data" / "raw" / "olist_orders_dataset.csv").exists():
            return d
    return start

ROOT = find_project_root()
RAW = ROOT / "data" / "raw"
CLEANED = ROOT / "data" / "preprocessed"
CLEANED.mkdir(parents=True, exist_ok=True)

print("Project root    :", ROOT)
print("Raw input       :", RAW)
print("Clean output    :", CLEANED)

assert RAW.exists(), f"raw/ not found. Expected: {RAW}"

pd.set_option("display.max_columns", None)


Notebook folder : E:\NUS最后一学期\IT5006\小组作业数据集\Clean data
Raw input       : E:\NUS最后一学期\IT5006\小组作业数据集\Clean data\raw
Clean output    : E:\NUS最后一学期\IT5006\小组作业数据集\Clean data\cleaned


## The 9 source tables

| # | Table | File | Grain (one row per &hellip;) | Rows |
|---|---|---|---|---|
| 1 | orders | olist_orders_dataset.csv | order | 99,441 |
| 2 | order_items | olist_order_items_dataset.csv | (order, item) | 112,650 |
| 3 | payments | olist_order_payments_dataset.csv | payment record | 103,886 |
| 4 | reviews | olist_order_reviews_dataset.csv | review record | 99,224 |
| 5 | customers | olist_customers_dataset.csv | customer_id | 99,441 |
| 6 | sellers | olist_sellers_dataset.csv | seller | 3,095 |
| 7 | products | olist_products_dataset.csv | product | 32,951 |
| 8 | geolocation | olist_geolocation_dataset.csv | geolocation record | 1,000,163 |
| 9 | translation | product_category_name_translation.csv | category name | 71 |

Eight tables are processed **independently** and **losslessly** (same rows in,
same rows out). `geolocation` is the one exception: 261,831 of its 1,000,163 rows
(26.18%) are byte-identical duplicates, so it is deduplicated to **738,332 unique
rows** &mdash; every unique `(zip, lat, lng, city, state)` combination is kept
exactly once.

In [2]:
# Schema contract for the 9 tables. dtype markers: str | int | float | datetime | category

BR_STATES = {
    "AC", "AL", "AM", "AP", "BA", "CE", "DF", "ES", "GO", "MA",
    "MG", "MS", "MT", "PA", "PB", "PE", "PI", "PR", "RJ", "RN",
    "RO", "RR", "RS", "SC", "SE", "SP", "TO",
}
ORDER_STATUS = {
    "approved", "canceled", "created", "delivered", "invoiced",
    "processing", "shipped", "unavailable",
}
PAYMENT_TYPE = {"boleto", "credit_card", "debit_card", "not_defined", "voucher"}
REVIEW_SCORE = {1, 2, 3, 4, 5}

TABLES = {
    "orders": {
        "file": "olist_orders_dataset.csv",
        "dtypes": {
            "order_id": "str", "customer_id": "str",
            "order_status": "category",
            "order_purchase_timestamp": "datetime",
            "order_approved_at": "datetime",
            "order_delivered_carrier_date": "datetime",
            "order_delivered_customer_date": "datetime",
            "order_estimated_delivery_date": "datetime",
        },
        "categorical": {"order_status": ORDER_STATUS},
    },
    "order_items": {
        "file": "olist_order_items_dataset.csv",
        "dtypes": {
            "order_id": "str", "order_item_id": "int",
            "product_id": "str", "seller_id": "str",
            "shipping_limit_date": "datetime",
            "price": "float", "freight_value": "float",
        },
        "categorical": {},
    },
    "payments": {
        "file": "olist_order_payments_dataset.csv",
        "dtypes": {
            "order_id": "str", "payment_sequential": "int",
            "payment_type": "category", "payment_installments": "int",
            "payment_value": "float",
        },
        "categorical": {"payment_type": PAYMENT_TYPE},
    },
    "reviews": {
        "file": "olist_order_reviews_dataset.csv",
        "dtypes": {
            "review_id": "str", "order_id": "str",
            "review_score": "int",
            "review_comment_title": "str", "review_comment_message": "str",
            "review_creation_date": "datetime",
            "review_answer_timestamp": "datetime",
        },
        "categorical": {},
    },
    "customers": {
        "file": "olist_customers_dataset.csv",
        "dtypes": {
            "customer_id": "str", "customer_unique_id": "str",
            "customer_zip_code_prefix": "str",   # leading zeros!
            "customer_city": "str", "customer_state": "category",
        },
        "categorical": {"customer_state": BR_STATES},
    },
    "sellers": {
        "file": "olist_sellers_dataset.csv",
        "dtypes": {
            "seller_id": "str", "seller_zip_code_prefix": "str",  # leading zeros!
            "seller_city": "str", "seller_state": "category",
        },
        "categorical": {"seller_state": BR_STATES},
    },
    "products": {
        "file": "olist_products_dataset.csv",
        "dtypes": {
            "product_id": "str", "product_category_name": "str",
            "product_name_lenght": "float", "product_description_lenght": "float",
            "product_photos_qty": "float", "product_weight_g": "float",
            "product_length_cm": "float", "product_height_cm": "float",
            "product_width_cm": "float",
        },
        "categorical": {},
    },
    "geolocation": {
        "file": "olist_geolocation_dataset.csv",
        "dtypes": {
            "geolocation_zip_code_prefix": "str",   # leading zeros!
            "geolocation_lat": "float", "geolocation_lng": "float",
            "geolocation_city": "str", "geolocation_state": "category",
        },
        "categorical": {"geolocation_state": BR_STATES},
        "deduplicate": True,   # 26.18% of rows are byte-identical duplicates
    },
    "translation": {
        "file": "product_category_name_translation.csv",
        "dtypes": {
            "product_category_name": "str",
            "product_category_name_english": "str",
        },
        "categorical": {},
    },
}

## What &ldquo;clean&rdquo; means here

1. **Type every column.** Dates are parsed to `datetime` (so they sort and filter
   correctly); `order_id`, `customer_id`, `product_id`, &hellip; stay strings; prices
   and measurements become numbers; low-cardinality enums become validated categories.
2. **Preserve leading zeros in zip codes.** `customer_zip_code_prefix`,
   `seller_zip_code_prefix` and `geolocation_zip_code_prefix` are read as **strings**,
   never integers.
3. **Validate, but never drop.** Categorical values are checked against a whitelist
   (states, order status, payment type, review score). Anything unexpected is
   **flagged in the log but kept** &mdash; we do not silently delete data.
4. **Lossless &mdash; except geolocation dedup.** For 8 tables the cleaned row count
   is verified to equal the raw row count. For `geolocation`, byte-identical
   duplicate rows (26.18%) are removed, and the cleaned count is verified to equal
   the number of *unique* raw rows.

### What we deliberately do *not* do

- &cross; merge tables &mdash; the 9 tables stay 9 separate tables;
- &cross; drop rows (incomplete first/last months, out-of-Brazil coordinates,
  `payment_value == 0` records, duplicate review ids &mdash; all kept). The *only*
  rows removed anywhere are `geolocation`&rsquo;s exact full-row duplicates;
- &cross; impute missing values &mdash; missing stays empty;
- &cross; add derived columns (revenue, on-time flags, &hellip;) &mdash; this is a
  *cleaning* step only; derived features belong to a later modelling stage.

In [3]:
def clean_table(name: str, spec: dict) -> pd.DataFrame:
    """Read one raw table, type + validate it, and return it cleaned."""
    dtypes = spec["dtypes"]
    date_cols = [c for c, t in dtypes.items() if t == "datetime"]
    read_dtype = {
        c: {"str": "string", "int": "int64", "float": "float64",
            "category": "category"}[t]
        for c, t in dtypes.items() if t != "datetime"
    }

    df = pd.read_csv(RAW / spec["file"], dtype=read_dtype, parse_dates=date_cols)

    # geolocation: drop byte-identical duplicate rows (26.18% of the table)
    if spec.get("deduplicate"):
        n_before = len(df)
        df = df.drop_duplicates()
        print(f"  [dedup] {name}: removed {n_before - len(df):,} "
              f"exact-duplicate rows ({n_before:,} -> {len(df):,})")

    # validate categorical domains (flag only -- never drop rows)
    for c, allowed in spec.get("categorical", {}).items():
        bad = set(df[c].dropna().unique()) - set(allowed)
        if bad:
            print(f"  [!] {name}.{c}: unexpected values {sorted(bad)} (kept)")

    # categories -> plain string for clean, tool-agnostic CSV output
    for c, t in dtypes.items():
        if t == "category":
            df[c] = df[c].astype("string")

    return df[list(dtypes.keys())]   # canonical column order

In [4]:
# read raw -> clean -> write cleaned CSV
summary = []
for name, spec in TABLES.items():
    df = clean_table(name, spec)
    out = CLEANED / spec["file"]
    df.to_csv(out, index=False, encoding="utf-8")
    summary.append((name, spec["file"], len(df)))
    print(f"[OK] {spec['file']:38s} {len(df):>9,} rows -> cleaned/")

summary_df = pd.DataFrame(summary, columns=["table", "file", "rows"])
summary_df


[OK] olist_orders_dataset.csv                  99,441 rows -> cleaned/


[OK] olist_order_items_dataset.csv            112,650 rows -> cleaned/


[OK] olist_order_payments_dataset.csv         103,886 rows -> cleaned/


[OK] olist_order_reviews_dataset.csv           99,224 rows -> cleaned/


[OK] olist_customers_dataset.csv               99,441 rows -> cleaned/
[OK] olist_sellers_dataset.csv                  3,095 rows -> cleaned/


[OK] olist_products_dataset.csv                32,951 rows -> cleaned/


  [dedup] geolocation: removed 261,831 exact-duplicate rows (1,000,163 -> 738,332)


[OK] olist_geolocation_dataset.csv            738,332 rows -> cleaned/
[OK] product_category_name_translation.csv         71 rows -> cleaned/


,table,file,rows
0,orders,olist_orders_dataset.csv,99441
1,order_items,olist_order_items_dataset.csv,112650
2,payments,olist_order_payments_dataset.csv,103886
3,reviews,olist_order_reviews_dataset.csv,99224
4,customers,olist_customers_dataset.csv,99441
5,sellers,olist_sellers_dataset.csv,3095
6,products,olist_products_dataset.csv,32951
7,geolocation,olist_geolocation_dataset.csv,738332
8,translation,product_category_name_translation.csv,71


In [5]:
# Row-count check: 8 tables lossless (clean == raw); geolocation deduped (clean == unique raw).
# usecols=[0] so quoted newlines inside review comments are safe.
print("Row-count verification (raw vs cleaned)")
all_ok = True
for name, spec in TABLES.items():
    raw_n = pd.read_csv(RAW / spec["file"], usecols=[0]).shape[0]
    clean_n = pd.read_csv(CLEANED / spec["file"], usecols=[0]).shape[0]
    if spec.get("deduplicate"):
        raw_unique = pd.read_csv(RAW / spec["file"]).drop_duplicates().shape[0]
        ok = clean_n == raw_unique
        print(f"  [{'OK ' if ok else 'FAIL'}] {spec['file']:38s} "
              f"raw={raw_n:>9,}  unique={raw_unique:>9,}  clean={clean_n:>9,}")
    else:
        ok = raw_n == clean_n
        print(f"  [{'OK ' if ok else 'FAIL'}] {spec['file']:38s} "
              f"raw={raw_n:>9,}  clean={clean_n:>9,}")
    all_ok &= ok

print("\nRESULT:", "ALL CHECKS PASSED" if all_ok else "ROW MISMATCH FOUND")

Row-count verification (raw vs cleaned)


  [OK ] olist_orders_dataset.csv               raw=   99,441  clean=   99,441


  [OK ] olist_order_items_dataset.csv          raw=  112,650  clean=  112,650


  [OK ] olist_order_payments_dataset.csv       raw=  103,886  clean=  103,886


  [OK ] olist_order_reviews_dataset.csv        raw=   99,224  clean=   99,224
  [OK ] olist_customers_dataset.csv            raw=   99,441  clean=   99,441
  [OK ] olist_sellers_dataset.csv              raw=    3,095  clean=    3,095


  [OK ] olist_products_dataset.csv             raw=   32,951  clean=   32,951


  [OK ] olist_geolocation_dataset.csv          raw=1,000,163  unique=  738,332  clean=  738,332
  [OK ] product_category_name_translation.csv  raw=       71  clean=       71

RESULT: ALL CHECKS PASSED


In [6]:
# zip-code prefixes kept their leading zeros
for name, col in [("customers", "customer_zip_code_prefix"),
                  ("sellers", "seller_zip_code_prefix"),
                  ("geolocation", "geolocation_zip_code_prefix")]:
    z = pd.read_csv(CLEANED / TABLES[name]["file"], dtype={col: str})[col]
    n0 = z.str.startswith("0").sum()
    samples = z[z.str.startswith("0")].head(2).tolist()
    print(f"{name}.{col}: {n0:>7,} values begin with '0'  e.g. {samples}")


customers.customer_zip_code_prefix:  23,995 values begin with '0'  e.g. ['09790', '01151']
sellers.seller_zip_code_prefix:   1,027 values begin with '0'  e.g. ['04195', '01529']


geolocation.geolocation_zip_code_prefix: 158,832 values begin with '0'  e.g. ['01037', '01046']


In [7]:
# preview cleaned orders (ISO dates, status as text, no index)
pd.read_csv(CLEANED / "olist_orders_dataset.csv").head()


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26
